# Phi Analysis: A Reproducible Notebook Example

This notebook demonstrates how Jupyter supports the paper workflow without replacing the project's source code. It reuses the public implementation, records the Git revision and experiment settings, runs a small morphology analysis, and exports paper-ready figures to an ignored experiment directory.

The original file at this path contained early notes about GNN dynamics, MBRL, node features, and morphology modes, but it was plain text with an .ipynb extension. Those ideas evolved into the current scalar-Phi hierarchy described in PAPER_DIRECTION.md.

## 1. Reproducibility metadata

Every paper experiment should identify the code revision, configuration, random seed, and output location. This makes a plot traceable months later.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Start Jupyter from the robot-sim repository root.")

sys.path.insert(0, str(PROJECT_ROOT))
from scripts.collect_gnn_data import compute_angles_from_phi
from src.utils.graph_converter import RobotGraphConverter

OUTPUT_DIR = PROJECT_ROOT / "experiments" / "notebook_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def git_output(*args):
    result = subprocess.run(
        ["git", *args], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True
    )
    return result.stdout.strip()

CONFIG = {
    "experiment": "phi_mapping_demo",
    "phi_min": 0.0,
    "phi_max": 3.0,
    "num_samples": 301,
    "seed": 20260907,
}

run_metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_output("rev-parse", "HEAD"),
    "git_dirty": bool(git_output("status", "--porcelain")),
    "python": sys.version.split()[0],
    "config": CONFIG,
}

metadata_path = OUTPUT_DIR / "run_metadata.json"
metadata_path.write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
run_metadata

## 2. Sweep the implemented morphology coordinate

The notebook calls the same analytical mapping used by the training and simulation scripts. No formula is duplicated here.

In [ ]:
np.random.seed(CONFIG["seed"])
phis = np.linspace(CONFIG["phi_min"], CONFIG["phi_max"], CONFIG["num_samples"])
joint_targets = np.asarray([compute_angles_from_phi(phi) for phi in phis])

assert joint_targets.shape == (CONFIG["num_samples"], 7)
assert np.isfinite(joint_targets).all()

{
    "shape": joint_targets.shape,
    "minimum_angle_rad": float(joint_targets.min()),
    "maximum_angle_rad": float(joint_targets.max()),
}

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)

colors = plt.cm.tab10(np.linspace(0, 0.9, 7))
for joint_index, color in enumerate(colors, start=1):
    ax.plot(
        phis, joint_targets[:, joint_index - 1],
        color=color, linewidth=2.0, label=f"Joint {joint_index}"
    )

for boundary in (1.0, 2.0):
    ax.axvline(boundary, color="#555555", linewidth=1.0, linestyle="--")

ax.set(
    title="Analytical Joint Targets Along the Phi Morphology Path",
    xlabel="Morphology coordinate Phi",
    ylabel="Target angle (rad)",
    xlim=(0.0, 3.0),
)
ax.legend(ncol=2, frameon=True)

figure_path = OUTPUT_DIR / "phi_joint_targets.png"
fig.savefig(figure_path, dpi=220, bbox_inches="tight")
plt.show()
figure_path

## 3. Inspect the graph contract and lock constraints

This check connects a paper statement to an executable invariant: the MorphGNN input always has seven nodes and ten features, and locked nodes carry their enforced base angle.

In [ ]:
converter = RobotGraphConverter()
probe_phis = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
lock_matrix = []

for phi in probe_phis:
    features = converter.get_feature_matrix(float(phi))
    assert features.shape == (7, 10)
    assert np.allclose(features[:, 7], phi / 3.0)
    lock_matrix.append(features[:, 8])

lock_matrix = np.asarray(lock_matrix)

fig, ax = plt.subplots(figsize=(8.5, 3.6), constrained_layout=True)
image = ax.imshow(lock_matrix, cmap="Greys", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(7), converter.MORPH_GRAPH_NODE_NAMES)
ax.set_yticks(range(len(probe_phis)), [f"{p:.1f}" for p in probe_phis])
ax.set_xlabel("Morphology graph node")
ax.set_ylabel("Phi")
ax.set_title("Implemented Joint-Lock Schedule")
fig.colorbar(image, ax=ax, ticks=[0, 1], label="Lock state")

lock_figure_path = OUTPUT_DIR / "phi_lock_schedule.png"
fig.savefig(lock_figure_path, dpi=220, bbox_inches="tight")
plt.show()
lock_figure_path

## 4. How this helps the paper and Git workflow

- Put reusable algorithms in src/ or scripts/, then import them here.
- Use the notebook for inspection, plots, tables, and short explanations.
- Record the Git commit and experiment configuration before drawing conclusions.
- Keep generated figures, metadata, datasets, and checkpoints under experiments/; this repository ignores that directory.
- Clear notebook outputs before committing so binary image output does not obscure code review.
- Promote a stable analysis into a tested script when it becomes part of the formal evaluation pipeline.

To run the example, start jupyter lab from the repository root, open this file, and choose Run > Run All Cells. The two figures and the metadata manifest will appear under experiments/notebook_demo/.